# Normalização de colunas no banco

Funções auxiliares para aplicar fatores de normalização diretamente em tabelas PostgreSQL usando SQLAlchemy.

In [ ]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, Integer, SmallInteger



import os
from pathlib import Path


from sql_utils import drop_table, build_postgres_engine


project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero

In [2]:
def normalize_table_columns_by_factor(
    engine: Engine,
    schema_name: str,
    table_name: str,
    columns: list[str],
    factor: int | float,
    filters: dict[str, object] | None = None,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    values = {}
    integer_types = (SmallInteger, Integer, BigInteger)

    for column_name in columns:
        column = table.c[column_name]
        normalized_value = column * factor

        if isinstance(column.type, integer_types):
            normalized_value = cast(func.trunc(normalized_value), column.type)

        values[column_name] = normalized_value

    statement = update(table).values(**values)

    if filters:
        for filter_column_name, filter_value in filters.items():
            statement = statement.where(table.c[filter_column_name] == filter_value)

    with engine.begin() as connection:
        connection.execute(statement)


# Ajuste para num absolutos

### Tabelas em Numeros já absolutos:

raw/APM_Demografico

raw/SENASP

raw/Pib_municipal

raw/dist_idade

raw/dist_renda_ibge

### Não se aplica:

raw/dist_instrucao_genero

raw/SINAN_Violencia

## Ajuste de raw/dist_instrução

In [ ]:
schema_name = "raw"
table_name = "dist_instrucao"

columns = [
    "Total",
    "Sem instrução e menos de 1 ano de estudo",
    "Ensino fundamental incompleto ou equivalente",
    "Ensino fundamental completo ou equivalente",
    "Ensino médio incompleto ou equivalente",
    "Ensino médio completo ou equivalente",
    "Ensino superior incompleto ou equivalente",
    "Ensino superior completo ou equivalente",
    "Não determinado",
]

normalize_table_columns_by_factor(
    engine,
    schema_name,
    table_name,
    columns,
    1000,
)
